In [1]:
!pip install -q gradio plotly ta google-genai kagglehub

  Preparing metadata (setup.py) ... done


In [2]:
import kagglehub

path = kagglehub.dataset_download(
    "soachishti/pakistan-stock-exchange-psx-complete-dataset"
)

print(path)

100%|██████████| 8.78M/8.78M [00:00<00:00, 27.7MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1


In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import gradio as gr

from ta.trend import SMAIndicator
from ta.momentum import RSIIndicator
from ta.trend import MACD

from google import genai

In [7]:
import glob

csv_files = glob.glob(f"{path}/**/*.csv", recursive=True)

print(csv_files)

['/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data/lot.csv', '/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data/sector.csv', '/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data/historical/alnrs.csv', '/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data/historical/modam.csv', '/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data/historical/aabs.csv', '/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data/historical/pinl.csv', '/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data/historical/shcm.csv', '/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data/historical/hubc.csv', '/

In [14]:
import os

for root, dirs, files in os.walk(path):
    print(root)
    print(files)
    print("-"*50)

/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1
[]
--------------------------------------------------
/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data
['lot.csv', 'sector.csv']
--------------------------------------------------
/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data/historical
['alnrs.csv', 'modam.csv', 'aabs.csv', 'pinl.csv', 'shcm.csv', 'hubc.csv', 'gvgl.csv', 'jatm.csv', 'ncl.csv', 'nagc.csv', 'ahcl.csv', 'thccl.csv', 'agsml.csv', 'anl.csv', 'pim.csv', 'spl.csv', 'pso.csv', 'colg.csv', 'kml.csv', 'dcr.csv', 'csil.csv', 'fudlm.csv', 'htl.csv', 'cppl.csv', 'elcm.csv', 'macfl.csv', 'abl.csv', 'ccm.csv', 'sutm.csv', 'gati.csv', 'lpgl.csv', 'nons.csv', 'searl.csv', 'flyng.csv', 'rpl.csv', 'pnsc.csv', 'khtc.csv', 'prl.csv', 'pibtl.csv', 'fham.csv', 'dwtm.csv', 'ticl.csv', 'ffl.csv', 'merit.csv', 'bwcl.csv', 'nestle.csv',

In [18]:
import pandas as pd

lots_path = f"{path}/data/lot.csv"
print (lots_path)

lots_df = pd.read_csv(lots_path)

print(lots_df.head())

print(lots_df.columns)

/root/.cache/kagglehub/datasets/soachishti/pakistan-stock-exchange-psx-complete-dataset/versions/1/data/lot.csv
  market security_ticker  lot
0    REG             786  500
1    REG            AABS  100
2    REG             AAL   50
3    REG            AASM  500
4    REG            AATM  500
Index(['market', 'security_ticker', 'lot'], dtype='object')


In [22]:
lots_df.columns = [
    c.lower().strip()
    for c in lots_df.columns
]

print(lots_df.columns)
symbols = sorted(
    lots_df['security_ticker']
    .dropna()
    .unique()
    .tolist()
)

print(symbols[:20])
historical_path = f"{path}/data/historical"

print(os.listdir(historical_path)[:10])

Index(['market', 'security_ticker', 'lot'], dtype='object')
['786', 'AABS', 'AAL', 'AASM', 'AATM', 'ABL', 'ABOT', 'ABSON', 'ACPL', 'ADAMS', 'ADMM', 'ADOS', 'AEL', 'AGIC', 'AGIL', 'AGL', 'AGLNCPS', 'AGP', 'AGSML', 'AGTL']
['alnrs.csv', 'modam.csv', 'aabs.csv', 'pinl.csv', 'shcm.csv', 'hubc.csv', 'gvgl.csv', 'jatm.csv', 'ncl.csv', 'nagc.csv']


In [23]:
import os

def load_stock_data(symbol):

    file_path = os.path.join(
        historical_path,
        f"{symbol}.csv"
    )
    print (file_path)

    if not os.path.exists(file_path):
        return None

    stock_df = pd.read_csv(file_path)

    stock_df.columns = [
        c.lower().strip()
        for c in stock_df.columns
    ]

    # Detect date column automatically
    for col in stock_df.columns:
        if 'date' in col:
            date_col = col
            break

    stock_df[date_col] = pd.to_datetime(
        stock_df[date_col]
    )

    stock_df = stock_df.sort_values(date_col)

    return stock_df

In [25]:
test_df = load_stock_data("786")

print(test_df.head())

print(test_df.columns)

UnboundLocalError: cannot access local variable 'date_col' where it is not associated with a value

In [ ]:
def add_indicators(stock_df):

    stock_df = stock_df.copy()

    stock_df['sma20'] = SMAIndicator(
        close=stock_df['close'],
        window=20
    ).sma_indicator()

    stock_df['sma50'] = SMAIndicator(
        close=stock_df['close'],
        window=50
    ).sma_indicator()

    stock_df['rsi'] = RSIIndicator(
        close=stock_df['close'],
        window=14
    ).rsi()

    macd = MACD(stock_df['close'])

    stock_df['macd'] = macd.macd()
    stock_df['macd_signal'] = macd.macd_signal()

    return stock_df

In [ ]:
def create_candlestick(stock_df):

    fig = go.Figure()

    fig.add_trace(
        go.Candlestick(
            x=stock_df['date'],
            open=stock_df['open'],
            high=stock_df['high'],
            low=stock_df['low'],
            close=stock_df['close'],
            name='Price'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=stock_df['date'],
            y=stock_df['sma20'],
            mode='lines',
            name='SMA20'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=stock_df['date'],
            y=stock_df['sma50'],
            mode='lines',
            name='SMA50'
        )
    )

    fig.update_layout(
        title="Price Analysis",
        xaxis_rangeslider_visible=False,
        height=600
    )

    return fig

In [ ]:
def create_volume_chart(stock_df):

    fig = px.bar(
        stock_df,
        x='date',
        y='volume',
        title='Volume Analysis'
    )

    return fig

In [ ]:
def create_rsi_chart(stock_df):

    fig = px.line(
        stock_df,
        x='date',
        y='rsi',
        title='RSI Indicator'
    )

    return fig

In [ ]:
from google import genai
import os

os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"
client = genai.Client()
def generate_ai_analysis(symbol, stock_df):

    latest = stock_df.iloc[-1]

    prompt = f"""
    Analyze this Pakistan stock.

    Symbol: {symbol}

    Latest Close: {latest['close']}

    RSI: {latest['rsi']}

    MACD: {latest['macd']}

    Give:
    1. Trend analysis
    2. Risk level
    3. Momentum
    4. Technical interpretation
    5. Long term outlook
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [ ]:
def analyze_stock(symbol):

    stock_df = df[df['symbol'] == symbol].copy()

    stock_df = add_indicators(stock_df)

    candle = create_candlestick(stock_df)

    volume = create_volume_chart(stock_df)

    rsi = create_rsi_chart(stock_df)

    ai_analysis = generate_ai_analysis(
        symbol,
        stock_df
    )

    latest_price = stock_df.iloc[-1]['close']

    sector = stock_df.iloc[-1]['sector']

    summary = f"""
    Symbol: {symbol}

    Sector: {sector}

    Latest Price: {latest_price}
    """

    return (
        summary,
        candle,
        volume,
        rsi,
        ai_analysis
    )

In [ ]:
def sector_analysis():

    latest = df.sort_values('date').groupby('symbol').tail(1)

    sector_perf = latest.groupby('sector')['close'].mean()

    sector_perf = sector_perf.reset_index()

    fig = px.treemap(
        sector_perf,
        path=['sector'],
        values='close',
        color='close',
        title='Sector Heatmap'
    )

    return fig

In [ ]:
def compare_stocks(sym1, sym2):

    s1 = df[df['symbol'] == sym1]
    s2 = df[df['symbol'] == sym2]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=s1['date'],
            y=s1['close'],
            name=sym1
        )
    )

    fig.add_trace(
        go.Scatter(
            x=s2['date'],
            y=s2['close'],
            name=sym2
        )
    )

    fig.update_layout(
        title="Stock Comparison"
    )

    return fig

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# PSX AI Analytics Platform")

    with gr.Tab("Stock Analysis"):

        symbol_input = gr.Dropdown(
            choices=symbols,
            label="Select Symbol"
        )

        analyze_btn = gr.Button("Analyze")

        summary_output = gr.Textbox(label="Summary")

        candle_output = gr.Plot()

        volume_output = gr.Plot()

        rsi_output = gr.Plot()

        ai_output = gr.Markdown(label="AI Analysis")

        analyze_btn.click(
            analyze_stock,
            inputs=symbol_input,
            outputs=[
                summary_output,
                candle_output,
                volume_output,
                rsi_output,
                ai_output
            ]
        )

    with gr.Tab("Sector Analysis"):

        sector_plot = gr.Plot(
            value=sector_analysis
        )

    with gr.Tab("Compare Stocks"):

        stock1 = gr.Dropdown(
            choices=symbols,
            label="Stock 1"
        )

        stock2 = gr.Dropdown(
            choices=symbols,
            label="Stock 2"
        )

        compare_btn = gr.Button("Compare")

        compare_plot = gr.Plot()

        compare_btn.click(
            compare_stocks,
            inputs=[stock1, stock2],
            outputs=compare_plot
        )

In [ ]:
demo.launch(share=True, debug=True)